# 01 · Знакомство с Olist

**Вопрос.** Что именно мы считаем, когда считаем строки? В заказе может быть несколько товаров, способов оплаты и отзывов. До первого объединения выясним, какая запись уникальна в каждом файле.

Все ячейки выполняются сверху вниз. Нужны девять исходных CSV в папке `data/`.

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"

files = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category": "product_category_name_translation.csv",
}

dates = {
    "orders": ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
               "order_delivered_customer_date", "order_estimated_delivery_date"],
    "items": ["shipping_limit_date"],
    "reviews": ["review_creation_date", "review_answer_timestamp"],
}
zip_codes = {f"{name}_zip_code_prefix": "string" for name in ["customer", "seller", "geolocation"]}
tables = {}
for name, filename in files.items():
    tables[name] = pd.read_csv(DATA / filename, dtype=zip_codes, parse_dates=dates.get(name))

## Девять таблиц и единица наблюдения

Почтовый префикс читаем как строку, чтобы сохранить ведущие нули. Для размеров считаем все исходные строки, включая полные повторы.

In [2]:
grain = {
    "orders": "заказ",
    "customers": "запись покупателя для конкретного заказа",
    "items": "одна товарная позиция в заказе",
    "payments": "одна часть оплаты заказа",
    "reviews": "отзыв, связанный с заказом",
    "products": "карточка товара",
    "sellers": "продавец",
    "geolocation": "наблюдение координат почтового префикса",
    "category": "перевод названия категории",
}
inventory = []
for name, df in tables.items():
    inventory.append([files[name], grain[name], len(df), df.shape[1], df.duplicated().sum()])

pd.DataFrame(inventory, columns=["Файл", "Одна строка", "Строк", "Столбцов", "Полных повторов"])

,Файл,Одна строка,Строк,Столбцов,Полных повторов
0,olist_orders_dataset.csv,заказ,99441,8,0
1,olist_customers_dataset.csv,запись покупателя для конкретного заказа,99441,5,0
2,olist_order_items_dataset.csv,одна товарная позиция в заказе,112650,7,0
3,olist_order_payments_dataset.csv,одна часть оплаты заказа,103886,5,0
4,olist_order_reviews_dataset.csv,"отзыв, связанный с заказом",99224,7,0
5,olist_products_dataset.csv,карточка товара,32951,9,0
6,olist_sellers_dataset.csv,продавец,3095,4,0
7,olist_geolocation_dataset.csv,наблюдение координат почтового префикса,1000163,5,261831
8,product_category_name_translation.csv,перевод названия категории,71,2,0


In [3]:
# Одна строка на столбец. Для идентификаторов и справочников полезна кардинальность.
profiles = []
for name, df in tables.items():
    profile = pd.DataFrame({
        "тип": df.dtypes.astype(str),
        "пропуски": df.isna().sum(),
        "различных непустых": df.nunique(),
    })
    profile.insert(0, "таблица", name)
    profiles.append(profile.rename_axis("столбец").reset_index())

with pd.option_context("display.max_rows", 100):
    display(pd.concat(profiles, ignore_index=True))

,столбец,таблица,тип,пропуски,различных непустых
0,order_id,orders,str,0,99441
1,customer_id,orders,str,0,99441
2,order_status,orders,str,0,8
3,order_purchase_timestamp,orders,datetime64[us],0,98875
4,order_approved_at,orders,datetime64[us],160,90733
5,order_delivered_carrier_date,orders,datetime64[us],1783,81018
6,order_delivered_customer_date,orders,datetime64[us],2965,95664
7,order_estimated_delivery_date,orders,datetime64[us],0,459
8,customer_id,customers,str,0,99441
9,customer_unique_id,customers,str,0,96096


## Где ключ составной

`order_id` подходит для таблицы заказов. В позициях к нему нужен номер позиции, в оплатах — номер части платежа. Сравним кандидаты на ключ в остальных таблицах.

In [4]:
keys = [
    ("orders", ["order_id"]), ("customers", ["customer_id"]),
    ("customers", ["customer_unique_id"]), ("items", ["order_id", "order_item_id"]),
    ("payments", ["order_id", "payment_sequential"]), ("reviews", ["review_id"]),
    ("reviews", ["order_id"]), ("reviews", ["order_id", "review_id"]),
    ("products", ["product_id"]), ("sellers", ["seller_id"]),
    ("category", ["product_category_name"]), ("geolocation", ["geolocation_zip_code_prefix"]),
]
checks = []
for name, columns in keys:
    df = tables[name]
    checks.append([name, " + ".join(columns), df[columns].isna().any(axis=1).sum(),
                   df.duplicated(columns).sum()])
pd.DataFrame(checks, columns=["Таблица", "Проверяемый ключ", "Строк с пропуском", "Строк сверх первой"])

,Таблица,Проверяемый ключ,Строк с пропуском,Строк сверх первой
0,orders,order_id,0,0
1,customers,customer_id,0,0
2,customers,customer_unique_id,0,3345
3,items,order_id + order_item_id,0,0
4,payments,order_id + payment_sequential,0,0
5,reviews,review_id,0,814
6,reviews,order_id,0,551
7,reviews,order_id + review_id,0,0
8,products,product_id,0,0
9,sellers,seller_id,0,0


В `orders` все 99 441 идентификатор заказа уникальны. В `items` 112 650 позиций, поскольку покупатель может положить в заказ несколько товаров. Для части заказов, например отменённых, позиции отсутствуют.

В `customers` 99 441 разных `customer_id` и 96 096 разных `customer_unique_id`. Через первый соединяем заказ с адресной записью, по второму узнаём человека при следующем заказе. Если считать возвраты по первому идентификатору, каждый клиент окажется «новым».

В `reviews` повторяются и `review_id`, и `order_id` по отдельности. На этой выгрузке пара из двух полей уникальна. Для геолокации один префикс соответствует множеству наблюдений. Соединение с сырым справочником по префиксу размножит строки.

## Связи между таблицами

```text
customers ──customer_id── orders ──order_id──┬── items ──product_id── products ──category name── category
                                            │       └──seller_id─── sellers
                                            ├── payments
                                            └── reviews
customers и sellers ──почтовый префикс── множество наблюдений geolocation
```

Проверяем, сколько **строк** дочерней таблицы содержат непустой ключ, которого нет у родителя. Пустой ключ считаем отдельно. Название на английском получаем по пути `items → products → category`.

In [5]:
links = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("items", "order_id", "orders", "order_id"),
    ("items", "product_id", "products", "product_id"),
    ("items", "seller_id", "sellers", "seller_id"),
    ("payments", "order_id", "orders", "order_id"),
    ("reviews", "order_id", "orders", "order_id"),
    ("products", "product_category_name", "category", "product_category_name"),
    ("customers", "customer_zip_code_prefix", "geolocation", "geolocation_zip_code_prefix"),
    ("sellers", "seller_zip_code_prefix", "geolocation", "geolocation_zip_code_prefix"),
]
coverage = []
for child, key, parent, ref in links:
    values = tables[child][key]
    unmatched = values.notna() & ~values.isin(tables[parent][ref])
    coverage.append([f"{child}.{key}", f"{parent}.{ref}", values.isna().sum(), unmatched.sum()])
pd.DataFrame(coverage, columns=["Проверяемое поле", "Справочник", "Пусто", "Нет соответствия"])

,Проверяемое поле,Справочник,Пусто,Нет соответствия
0,orders.customer_id,customers.customer_id,0,0
1,items.order_id,orders.order_id,0,0
2,items.product_id,products.product_id,0,0
3,items.seller_id,sellers.seller_id,0,0
4,payments.order_id,orders.order_id,0,0
5,reviews.order_id,orders.order_id,0,0
6,products.product_category_name,category.product_category_name,610,13
7,customers.customer_zip_code_prefix,geolocation.geolocation_zip_code_prefix,0,278
8,sellers.seller_zip_code_prefix,geolocation.geolocation_zip_code_prefix,0,7


У 610 товаров категория пуста, ещё у 13 товаров с указанной категорией отсутствует перевод. Поэтому левое соединение со справочником сохраняет продажи даже при отсутствующем названии. Штат покупателя уже есть в `customers` и доступен, даже если его индекс отсутствует в геолокации.

## Статусы и даты

Сопоставим статус с датой получения. У заказа в пути поле доставки часто пустое. У заказа со статусом `delivered` тот же пропуск требует отдельного решения.

In [6]:
orders = tables["orders"]
print("Первая покупка:", orders["order_purchase_timestamp"].min())
print("Последняя покупка:", orders["order_purchase_timestamp"].max())

missing_dates = orders[dates["orders"]].isna().groupby(orders["order_status"]).sum()
missing_dates.insert(0, "всего заказов", orders["order_status"].value_counts())
missing_dates.sort_values("всего заказов", ascending=False)

Первая покупка: 2016-09-04 21:15:19
Последняя покупка: 2018-10-17 17:30:18


,всего заказов,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
order_status,,,,,,
delivered,96478,0,14,2,8,0
shipped,1107,0,0,0,1107,0
canceled,625,0,141,550,619,0
unavailable,609,0,0,609,609,0
invoiced,314,0,0,314,314,0
processing,301,0,0,301,301,0
created,5,0,5,5,5,0
approved,2,0,0,2,2,0


Покупки в выгрузке охватывают период с 4 сентября 2016 по 17 октября 2018 года. Из 96 478 доставленных заказов у 8 отсутствует дата вручения. Сам заказ учитывается в числе доставленных, а длительность и опоздание для него остаются неизвестными. При сравнении месяцев важно помнить, что на краях периода есть месяцы лишь с несколькими заказами.

## Какие решения можно проверить на этих данных

1. Как менялись оборот товаров и число доставленных заказов по месяцам?
2. В каких штатах покупатели приносят больше оборота?
3. Какие категории вносят наибольший вклад в продажи?
4. Насколько продажи внутри штата зависят от нескольких продавцов?
5. Какую долю покупателей мы видим с повторным заказом?
6. Как возвращаются покупатели разных когорт после первой покупки?
7. В каких штатах или у каких продавцов заказы чаще приезжают поздно?
8. Как меняется оценка заказа при опоздании?
9. Как стоимость доставки связана с суммой за товары?
10. Какие способы оплаты встречаются чаще и как различается размер заказа?